# Tutorial: CT-01 Categories and Diagrams

Audience:
- Learners in Module 00 who know basic set theory and linear algebra.

Prerequisites:
- Familiarity with functions, vectors, and matrix multiplication.
- Comfortable reading simple mathematical notation.

Learning goals:
- Read a simple compositional diagram as an equality of maps.
- Distinguish sequential composition from parallel structure.
- Recognize five ML constructions that naturally fit categorical language.


## Outline

1. Build a tiny diagram helper.
2. Review the standard categories `Set`, `Vect`, and `Grp`.
3. Render five ML constructions as compositional diagrams.
4. Note what category theory clarifies and what it does not replace.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class Arrow:
    source: str
    label: str
    target: str


def chain(*arrows: Arrow) -> str:
    if not arrows:
        return ""
    pieces = [arrows[0].source]
    for arrow in arrows:
        pieces.append(f" --{arrow.label}--> {arrow.target}")
    return "".join(pieces)


def composite_name(*labels: str) -> str:
    return " ∘ ".join(reversed(labels))


def square(top_left: str, top_arrow: str, top_right: str, left_arrow: str, right_arrow: str, bottom_left: str, bottom_arrow: str, bottom_right: str) -> str:
    return "\n".join(
        [
            f"{top_left} --{top_arrow}--> {top_right}",
            f"|{left_arrow}{' ' * max(1, len(top_arrow) - len(left_arrow) + 2)}|{right_arrow}",
            f"v{' ' * (len(top_left) + len(top_arrow) + 1)}v",
            f"{bottom_left} --{bottom_arrow}--> {bottom_right}",
        ]
    )


standard_categories = {
    "Set": {"objects": "sets", "morphisms": "functions"},
    "Vect_R": {"objects": "real vector spaces", "morphisms": "linear maps"},
    "Grp": {"objects": "groups", "morphisms": "group homomorphisms"},
}

standard_categories


## Standard categories first

Before moving to ML, keep the basic examples in view:

- In `Set`, morphisms are ordinary functions.
- In `Vect`, morphisms are linear maps, so matrix multiplication becomes composition.
- In `Grp`, morphisms must preserve multiplication, not just map elements arbitrarily.

The notebook examples below borrow this language but stay concrete.


## 1. Feature pipeline

A preprocessing stage, feature extractor, and classifier form a single composite map:

$$
X \xrightarrow{n} X' \xrightarrow{\phi} H \xrightarrow{c} Y.
$$

The full predictor is $c \circ \phi \circ n$.


In [ ]:
feature_pipeline = [
    Arrow("X", "n", "X'"),
    Arrow("X'", "phi", "H"),
    Arrow("H", "c", "Y"),
]

{
    "diagram": chain(*feature_pipeline),
    "composite": composite_name("n", "phi", "c"),
}


## 2. Training loop

A single training step is a map from parameters back to parameters:

$$
\Theta \xrightarrow{g_D} T\Theta \xrightarrow{u} \Theta.
$$

If $s_D = u \circ g_D$, then running multiple epochs means composing $s_D$ with itself repeatedly.


In [ ]:
training_step = [Arrow("Theta", "g_D", "TTheta"), Arrow("TTheta", "u", "Theta")]

{
    "diagram": chain(*training_step),
    "single_step": composite_name("g_D", "u"),
    "three_epochs": "s_D ∘ s_D ∘ s_D",
}


## 3. Encoder-decoder

An autoencoder factors a reconstruction through a latent object $Z$:

$$
X \xrightarrow{e} Z \xrightarrow{d} X.
$$

The reconstruction map is $d \circ e : X \to X$.


In [ ]:
encoder_decoder = [Arrow("X", "e", "Z"), Arrow("Z", "d", "X")]

{
    "diagram": chain(*encoder_decoder),
    "reconstruction": composite_name("e", "d"),
    "interpretation": "latent bottleneck makes the factorization explicit",
}


## 4. Data augmentation with label preservation

Augmentations compose like endomorphisms on the data space:

$$
X \xrightarrow{a_1} X \xrightarrow{a_2} X \xrightarrow{a_3} X.
$$

If labels are preserved, the square below commutes:

$$
\begin{CD}
X @>a>> X \\
@VyVV   @VVyV \\
\mathcal{Y} @= \mathcal{Y}
\end{CD}
$$

That equation is simply $y \circ a = y$.


In [ ]:
augmentation_chain = [Arrow("X", "a_1", "X"), Arrow("X", "a_2", "X"), Arrow("X", "a_3", "X")]
label_square = square("X", "a", "X", "y", "y", "Y", "id_Y", "Y")

{
    "augmentation_diagram": chain(*augmentation_chain),
    "label_preservation_square": label_square,
    "equation": "y ∘ a = y",
}


## 5. Transfer learning

A transferred model can be written as

$$
X_{\mathrm{src}} \xrightarrow{f_{\mathrm{src}}} H \xrightarrow{t} H' \xrightarrow{c_{\mathrm{tgt}}} Y_{\mathrm{tgt}}.
$$

The adaptation map $t$ represents the nontrivial step between a source representation and a target task.


In [ ]:
transfer_learning = [
    Arrow("X_src", "f_src", "H"),
    Arrow("H", "t", "H'"),
    Arrow("H'", "c_tgt", "Y_tgt"),
]

{
    "diagram": chain(*transfer_learning),
    "composite": composite_name("f_src", "t", "c_tgt"),
}


## Optional parallel view: encoder-decoder with a skip path

Primer mode only needs monoidal intuition, not the full formalism. A residual or multimodal system often has a sequential branch and an identity-like branch combined later. That is where categorical language starts to care about parallel composition in addition to ordinary composition.


In [ ]:
summary = {
    "standard_examples": list(standard_categories),
    "ml_constructions": [
        "feature pipeline",
        "training loop",
        "encoder-decoder",
        "data augmentation chain",
        "transfer learning",
    ],
    "main_question": "What are the objects, what are the morphisms, and which diagrams should commute?",
}

summary


## Exercises

- Replace the feature extractor $\phi$ with two parallel feature branches and describe what extra structure is needed.
- Write a commuting square for an equivariant data transformation instead of a label-preserving one.
- Modify the transfer-learning diagram so that the source and target tasks share the same representation object.
